# Decision Trees — Statistical Learning

A leveled exercise. Most topics can be solved at **three levels** — but only the **foundational** ones offer all three; later, applied topics jump straight to the practical level. Pick the level that fits your comfort.

## How this notebook works

<table>
<tr><td style="font-size:1.4em;">🧮</td><td><b style="color:#E37222;">Level 1 · By hand</b></td><td>Do the math yourself on small <b>mock data</b> — plain arithmetic, no libraries.</td></tr>
<tr><td style="font-size:1.4em;">🐍</td><td><b style="color:#0065BD;">Level 2 · With NumPy</b></td><td>Implement the formula with basic <b>NumPy</b> on the real data.</td></tr>
<tr><td style="font-size:1.4em;">📚</td><td><b style="color:#5E6A00;">Level 3 · With libraries</b></td><td>Use <b>pandas / scikit-learn</b> — the practical way.</td></tr>
</table>

When you get stuck, expand a hint: <b>💡 Hint</b> points you to the right functions (with docs); <b>🔑 Stuck?</b> shows them on tiny mock data.

## Setup &mdash; packages and data

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use(os.path.join(os.getcwd(), "..", "..", "configs", "visualisations.mplstyle"))
df = pd.read_csv(os.path.join("..", "..", "data", "1900_2021_DISASTERS.csv"))

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Disaster Type'] = df['Disaster Type'].astype('category')

types = ['Earthquake', 'Flood', 'Storm', 'Volcanic activity']
geo = df[df['Latitude'].between(-90, 90) & df['Longitude'].between(-180, 180)]
geo = geo[geo['Disaster Type'].isin(types)].copy()
X = geo[['Longitude', 'Latitude']].values
y = geo['Disaster Type'].astype(str).values
geo.head()

,Year,Seq,Glide,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,Disaster Subsubtype,Event Name,Country,...,No Affected,No Homeless,Total Affected,Insured Damages ('000 US$),Total Damages ('000 US$),CPI,Adm Level,Admin1 Code,Admin2 Code,Geo Locations
2,1902,12,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,Guatemala,...,NaN,NaN,NaN,NaN,25000.0,3.350513,NaN,NaN,NaN,NaN
9,1905,3,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,India,...,NaN,NaN,NaN,NaN,25000.0,3.479379,NaN,NaN,NaN,NaN
10,1906,14,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,Chile,...,NaN,NaN,NaN,NaN,100000.0,3.479379,NaN,NaN,NaN,NaN
15,1907,6,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,China,...,NaN,NaN,NaN,NaN,NaN,3.608245,NaN,NaN,NaN,NaN
21,1909,9,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,France,...,NaN,NaN,NaN,NaN,NaN,3.479379,NaN,NaN,NaN,NaN


# Learning Decision Trees

### Node impurity
Trees prefer splits that make regions **purer**. We measure purity with the **Gini index** $G = 1 - \sum_k p_k^2$, computed here three ways.

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">A node holds <b>3 Earthquakes and 1 Flood</b>. Compute its Gini index by hand and enter the value (recall p(EQ) = 3/4 and p(Flood) = 1/4).</span></div>

In [ ]:
# Mock node: 3 Earthquakes, 1 Flood

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print('Your Gini:', gini_by_hand)

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Write a function <code>gini(labels)</code> with NumPy that returns the Gini index 1 − Σ&nbsp;pₖ² of an array of class labels, and test it on the target <code>y</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Get counts with [`np.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)`(labels, return_counts=True)`, turn them into proportions, then [`np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)`(p**2)`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
mock = np.array(['a','a','a','b'])
_, c = np.unique(mock, return_counts=True)
p = c / c.sum()        # [0.75, 0.25]
g = 1 - np.sum(p**2)   # 0.375
```

</details>

In [ ]:
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print('Gini of the full dataset:', round(gini(y), 3))

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Fit a <code>DecisionTreeClassifier(max_depth=1)</code> and read the Gini impurity of its <b>root node</b> straight from the fitted tree — confirm it matches your function.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), then read `clf.tree_.impurity[0]` (element 0 is the root — see the [tree-structure example](https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html)).

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(max_depth=1).fit(X, y)
root_gini = clf.tree_.impurity[0]
```

</details>

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print('sklearn root Gini:', round(root_gini, 3))

### The best split — information gain

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement <code>info_gain(feature, labels, s)</code> = H(parent) − Σ_c (N_c/N)·H(child_c), where the entropy is H = −Σ&nbsp;pₖ·log₂(pₖ). Then sweep thresholds on <code>Latitude</code> for the split with the largest gain.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Reuse a NumPy `entropy(labels)` (like `gini`, but [`-np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)`(p*`[`np.log2`](https://numpy.org/doc/stable/reference/generated/numpy.log2.html)`(p))`); split with a boolean mask `feature <= s`; weight each child by its share of the rows.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
def entropy(lab):
    _, c = np.unique(lab, return_counts=True)
    p = c / c.sum()
    return -np.sum(p * np.log2(p))
left = feature <= s            # boolean mask
```

</details>

In [ ]:
feat = geo['Latitude'].values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

candidates = np.linspace(feat.min() + 1, feat.max() - 1, 200)
gains = [info_gain(feat, y, s) for s in candidates]
best_s = candidates[int(np.argmax(gains))]
print(f'best Latitude split: s = {best_s:.2f}  ->  IG = {max(gains):.4f}')
plt.plot(candidates, gains, color='#0065BD'); plt.axvline(best_s, color='#E37222', ls='--')
plt.xlabel('Latitude threshold s'); plt.ylabel('information gain'); plt.show()

### Fit and visualize a tree

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Train a <code>DecisionTreeClassifier(max_depth=3)</code> on the two location features, print its training accuracy, and draw it with <code>plot_tree</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), then draw it with [`plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)`(clf, feature_names=[...], class_names=clf.classes_, filled=True)`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
from sklearn.tree import plot_tree
plot_tree(clf, feature_names=['Longitude','Latitude'],
          class_names=clf.classes_, filled=True)
```

</details>

In [ ]:
from sklearn.tree import plot_tree

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'training accuracy: {clf.score(X, y):.3f}')
fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(clf, feature_names=['Longitude', 'Latitude'], class_names=clf.classes_,
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.show()

# Pruning Decision Trees

### Pre-pruning via depth

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Split into train/test, then sweep <code>max_depth</code> from 1 to 20 and plot training vs. test accuracy. Where does the tree start to overfit?</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Split with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)`(X, y, test_size=0.35, stratify=y)`; fit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) per depth and call `.score` on each split.

</details>

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=0, stratify=y)
depths = range(1, 21)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

plt.plot(list(depths), train_acc, '-o', ms=4, color='#005293', label='training')
plt.plot(list(depths), test_acc, '-o', ms=4, color='#E37222', label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend(); plt.show()

### Post-pruning via cost-complexity
Cost-complexity pruning scores a tree by error plus a size penalty: $R_\alpha(T)=R(T)+\alpha\,|T|$ ($|T|$ = number of leaves).

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">A full tree makes <b>3</b> errors with <b>4</b> leaves; pruned to the root it makes <b>8</b> errors with <b>1</b> leaf. With α = 2, compute R_α = errors + α·leaves for both and decide which to keep (smaller wins).</span></div>

In [ ]:
alpha = 2

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'R_full = {R_full}, R_root = {R_root}  ->  keep the {keep}')

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Compute R_α for both trees over a <b>range of α</b> and find the α at which pruning to the root becomes preferable.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Build `alphas = `[`np.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html)`(0, 5, 100)`; form `R_full`/`R_root` as arrays, then `alphas[`[`np.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)`(R_root < R_full)]`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
alphas = np.linspace(0, 5, 100)
a = 3 + alphas*4
b = 8 + alphas*1
crossover = alphas[np.argmax(b < a)]
```

</details>

In [ ]:
alphas = np.linspace(0, 5, 100)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'Prune to the root once alpha >= {crossover:.2f}')

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">On the real data, use <code>cost_complexity_pruning_path</code> to get the candidate α values, pick the α with the best test accuracy, and compare leaves of the pruned vs. unpruned tree.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

[`cost_complexity_pruning_path`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.cost_complexity_pruning_path)`(X_tr, y_tr).ccp_alphas` gives the candidates; refit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)`(ccp_alpha=a)` and score each, then [`np.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html).

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example on mock data</summary>

```python
path = base.cost_complexity_pruning_path(X_tr, y_tr)
for a in path.ccp_alphas[:-1]:
    DecisionTreeClassifier(ccp_alpha=a).fit(X_tr, y_tr)  # ... score ...
```

</details>

In [ ]:
base = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
alphas_path = base.cost_complexity_pruning_path(X_tr, y_tr).ccp_alphas[:-1]

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'best alpha = {best_alpha:.5f}')
print(f'leaves: {base.get_n_leaves()} -> {pruned.get_n_leaves()}')

# Popular Algorithms

### Gini vs. entropy

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">scikit-learn implements <b>CART</b> (always binary splits). Fit two trees with <code>criterion='gini'</code> and <code>criterion='entropy'</code> and compare their test accuracy — are they very different?</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Loop over the two criterion strings, fit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)`(criterion=crit, max_depth=6)`, and call `.score`.

</details>

In [ ]:
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print('\n(Both are CART trees — the criterion only changes how splits are scored.)')

# Tree Ensembles

### Bagging & out-of-bag error

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Train a <code>BaggingClassifier</code> of 200 trees with <code>oob_score=True</code> and compare its out-of-bag accuracy to a single tree's test accuracy.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

[`BaggingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)`(`[`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)`(), n_estimators=200, oob_score=True)`; then read `.oob_score_`.

</details>

In [ ]:
from sklearn.ensemble import BaggingClassifier

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'single tree test accuracy : {single.score(X_te, y_te):.3f}')
print(f'bagging OOB accuracy      : {bag.oob_score_:.3f}')
print(f'bagging test accuracy     : {bag.score(X_te, y_te):.3f}')

### Random forest & variable importance

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">On a richer feature set, fit a <code>RandomForestClassifier</code> and plot its <code>feature_importances_</code> to see which features best predict the disaster type.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit a [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html); `rf.feature_importances_` is one weight per feature. Sort with [`np.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) and draw with [`plt.barh`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.barh.html).

</details>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
feats = ['Longitude', 'Latitude', 'Total Deaths', 'No Affected', 'No Homeless', 'Start Month']
D = geo.copy()
for c in feats:
    D[c] = pd.to_numeric(D[c], errors='coerce')
Xr = D[feats].fillna(0).values
yr = D['Disaster Type'].astype(str).values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

order = np.argsort(importances)
plt.barh([feats[i] for i in order], importances[order], color='#0065BD')
plt.xlabel('feature importance (mean decrease in impurity)')
plt.title('Random forest — variable importance'); plt.show()

### Boosting

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Train a <code>GradientBoostingClassifier</code> and use <code>staged_predict</code> to plot training vs. test accuracy over the boosting rounds. Does it overfit?</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit a [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html); its `staged_predict(X)` yields the prediction after each round — wrap each in [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html).

</details>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.3, random_state=0).fit(X_tr, y_tr)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

plt.plot(range(1, 201), train_acc, color='#005293', label='training')
plt.plot(range(1, 201), test_acc, color='#E37222', label='test')
plt.xlabel('number of boosting rounds'); plt.ylabel('accuracy'); plt.legend(); plt.show()